In [42]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "brauer2005all")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Braeuer_2005_braeuer-gaze-2002.sav")
complete_path_2 = os.path.join(original_data_pathway, "Braeuer_2005_gaze-follow-data-all.sav")
complete_path_3 = os.path.join(original_data_pathway, "Braeuer_2005_braeuer-looking-around-2003.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [43]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
df2 = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)
df3 = pd.read_spss(complete_path_3, usecols=None, convert_categoricals=True)
experiment_import = [[df1, 'gaze', '1'],
                    [df2, 'gaze_all_summary', '1'],
                    [df3, 'looking_around','2']]
for x, y, k in experiment_import:
    x['experiment_name']=y
    x['experiment'] = k

# df3.columns


In [44]:
df3_temp1 = df3[['SESSION', 'TRIAL', 'SUBJECT', 'SPECIES', 'EXPER', 'REACT_EX',
        'SUC', 'experiment_name',
       'experiment']].values.tolist() + df3[[
              'SESSION', 'TRIAL', 'SUBJECT', 'SPECIES', 'CONTROL', 'REACT_CO', 
       'SUC', 'experiment_name',
       'experiment']].values.tolist()

df3_1 = pd.DataFrame(df3_temp1, columns=['SESSION', 'TRIAL', 'SUBJECT', 'SPECIES', 'condition', 'REACT_condition',
        'SUC', 'experiment_name',
       'experiment'])


In [45]:
data_frames=[df1, df2, df3_1]

for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"subject": "ape",
        "cond": "condition"})
    x['study_id']="brauer2005all"
    data_frames[index]=x
new_df=data_frames[0]

In [46]:
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

fulldf['ape'] = fulldf['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
# fulldf.columns

In [47]:
fulldf=fulldf.rename(columns={"sex_y": "sex",
    "species_y": "species",
    "age_grup": "age_group",
    "age_gaze":"age_original",
    "suc":'success'})

In [48]:
fulldf = fulldf[fulldf['ape'].notna()]
fulldf['session'].replace('', np.nan, inplace=True)
fulldf.rename(columns={"ape": "participant", 
    "react_condition":"duration_target_spot_in_seconds"}, inplace=True)



In [49]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age": "age_in_years"}, inplace=True) 

In [50]:
fulldf=fulldf[['study_id', 'experiment','experiment_name',
                'participant', 'age_original','age_in_years','sex', 'species',
        'session', 'trial', 'condition', 'reaction', 'success', 
       'age_group', 
       'gaze_ex', 'gaze_co', 'gaze_dif', 'double_check_ex', 'double_check_co',
        'duration_target_spot_in_seconds']]


In [51]:
exp1 = fulldf[fulldf['experiment_name'] == 'gaze']
exp2 = fulldf[fulldf['experiment_name'] == 'gaze_all_summary']
exp3 = fulldf[fulldf['experiment_name'] == 'looking_around']

# exp3=exp3.sort_values(['trial'])
exp3['trial']=exp3['trial'].astype(int)
exp3=exp3.sort_values(by = ['participant','trial'])
# exp3['trial'].unique

experiments = [[exp1, 'brauer2005all_exp1'],
                [ exp2, 'brauer2005all_exp1_summary'],
                [exp3, 'brauer2005all_exp2']]

for x,y in experiments:
    x = x.dropna(axis=1, how='all')## drop empty rows/columns
    comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
    x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
    ##glossaries
    names = x.columns.tolist()
    df = pd.DataFrame(names)
    df = df.rename(columns={0: "column_name"})
    df["description"] = ""
    studyID_glossary=df[["column_name", "description"]]

    comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
    studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

C:\Users\CARIN_~1\AppData\Local\Temp/ipykernel_1720/3112681823.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  exp3['trial']=exp3['trial'].astype(int)
